In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()



# ACC↔HPC cue-period multivariate MI (Frites GCMI, subject-time)

This notebook computes ACC↔HPC multivariate MI with **Frites GCMI** at each cue timepoint.
Output is **subject-time** (not trial-local), in **bits**.


In [ ]:

%matplotlib inline
%reload_ext autoreload
%autoreload 2

from pathlib import Path
from typing import Any, Dict, List, Tuple
import warnings
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from membandit.plotting_style import set_plot_style
from membandit.io import Subject
from membandit.granger import build_hit_miss_trial_map, build_rpe_trial_map
from mne.stats import permutation_cluster_test

warnings.filterwarnings('ignore')
set_plot_style()
sns.set_style('white')

# Fail fast if frites is not available in this kernel.
try:
    from frites.estimator import GCMIEstimator
except Exception as exc:
    raise ImportError(
        'Frites is not importable in this notebook kernel. Please install/activate the environment containing frites.'
    ) from exc


In [ ]:

# -----------------------------
# Config
# -----------------------------
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

EVENT = 'cue_on'
DAY = 'Day1'
ACC_ROI = 'ACC'
HPC_ROI = 'HPC'

FREQ_RANGE = (70.0, 201.0)
DOWNSAMPLE_FACTOR = 5
WIN_MS = 200
HFA_SMOOTH_METHOD = 'movmean'
HFA_ARTIFACT_ABS_THRESH = 10.0

SEED = 42
N_JOBS = -1
JOBLIB_BACKEND = 'loky'

# Min trials per class (hit or miss) when computing class-specific MI.
MIN_TRIALS_PER_CLASS = 5
N_PERM_CONTRAST = 200

N_PERM_CLUSTER = 5000
CLUSTER_ALPHA = 0.05

output_root = Path('/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/analysis_outputs/acc_hpc_cue_multivariate_mi')
fig_root = output_root / 'figures'
output_root.mkdir(parents=True, exist_ok=True)
fig_root.mkdir(parents=True, exist_ok=True)

print('Output root:', output_root)


In [ ]:

# -----------------------------
# Helpers
# -----------------------------
def _get_smoothed_event_signal(
    subject_obj: Subject,
    roi: str,
    event: str,
    freq_range: Tuple[float, float],
    downsample_factor: int,
    win_ms: int,
    smooth_method: str = 'movmean',
    artifact_abs_thresh: float = 10.0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    neural = subject_obj._get_or_build_event_neural_data(
        event=event,
        roi=roi,
        freq_range=freq_range,
        downsample_factor=downsample_factor,
        win_ms=win_ms,
        smooth_method=smooth_method,
        artifact_abs_thresh=artifact_abs_thresh,
    )
    signal = np.asarray(neural['hfa'], dtype=np.float64)
    times = np.asarray(neural['times'], dtype=np.float64)
    labels = np.asarray(neural.get('electrode_labels', []), dtype=str)
    if labels.size == 0:
        raise ValueError(f'Missing electrode_labels for {subject_obj.subject} {event}/{roi}')
    return signal, times, labels


def _resolve_trial_var_for_event(df: pd.DataFrame, event: str) -> str:
    if str(event).strip().lower() == 'cue_on':
        for candidate in ('trials_mem', 'trials_gamble', 'trials'):
            if candidate in df.columns:
                return candidate
    for candidate in ('trials', 'trials_gamble', 'trials_mem'):
        if candidate in df.columns:
            return candidate
    raise ValueError(f'Could not determine trial variable for event={event!r}')


def _build_cue_trial_axis(subject_obj: Subject, n_epochs: int, event: str = 'cue_on') -> Dict[str, Any]:
    cdf = subject_obj.combined_df[
        (subject_obj.combined_df.participant == subject_obj.subject)
        & (subject_obj.combined_df.condition == DAY)
    ].copy()
    if len(cdf) == 0:
        raise ValueError('no_day1_trials')

    trial_col = _resolve_trial_var_for_event(cdf, event)
    cdf = cdf.dropna(subset=[trial_col]).copy()
    trial_vals = pd.to_numeric(cdf[trial_col], errors='coerce').to_numpy(dtype=np.float64)
    in_range = np.isfinite(trial_vals) & ((trial_vals - 1) >= 0) & ((trial_vals - 1) < int(n_epochs))
    if int(np.sum(in_range)) == 0:
        raise ValueError('no_in_range_trials')

    trial_ix = (trial_vals[in_range].astype(np.int64) - 1)
    return {
        'trial_index': trial_ix.astype(np.int64, copy=False),
        'trial_number': trial_vals[in_range].astype(np.int64, copy=False),
        'trial_col': trial_col,
    }


def _build_behavior_labels_for_subject(subject_obj: Subject, n_epochs: int) -> pd.DataFrame:
    hm_map = build_hit_miss_trial_map(subject_obj._combined_df, subj_id=subject_obj.subject, n_epochs=int(n_epochs), day=DAY, event=EVENT)
    rpe_map = build_rpe_trial_map(subject_obj._combined_df, subj_id=subject_obj.subject, n_epochs=int(n_epochs), day=DAY, event=EVENT)

    if len(hm_map):
        hm_map = hm_map[hm_map['alignment_valid']].copy()
        hm = hm_map[['epoch_idx', 'hit_miss']].copy()
    else:
        hm = pd.DataFrame(columns=['epoch_idx', 'hit_miss'])

    if len(rpe_map):
        rpe_map = rpe_map[rpe_map['alignment_valid']].copy()
        rp = rpe_map[['epoch_idx', 'rpe_cat']].copy()
    else:
        rp = pd.DataFrame(columns=['epoch_idx', 'rpe_cat'])

    lab = hm.merge(rp, on='epoch_idx', how='outer')
    if len(lab) == 0:
        lab = pd.DataFrame({'epoch_idx': np.arange(int(n_epochs), dtype=int)})
    lab['subject'] = subject_obj.subject
    return lab


def _to_frites_mv(arr_trials_by_feat: np.ndarray) -> np.ndarray:
    # Frites expects (n_var, n_mv, n_samples). Here: one variable/timepoint.
    x = np.asarray(arr_trials_by_feat, dtype=np.float64)
    return np.ascontiguousarray(x.T[None, :, :])


def _safe_mi_cc(est: 'GCMIEstimator', X: np.ndarray, Y: np.ndarray, debug: bool = False) -> float:
    if X.shape[0] < 3:
        return np.nan
    xf = _to_frites_mv(X)
    yf = _to_frites_mv(Y)
    out = est.estimate(xf, yf)
    if debug:
        print('est.estimate return type:', type(out))
        if isinstance(out, (tuple, list)):
            print('est.estimate tuple/list length:', len(out))
            print('element types:', [type(k) for k in out])
    # Some versions may return tuple/list-like containers. Keep first item if so.
    if isinstance(out, (tuple, list)):
        out = out[0]
    return float(np.asarray(out).squeeze())


def _prepare_hfa_pair(subject_obj: Subject) -> Dict[str, Any]:
    acc_hfa, cue_times_acc, acc_labels = _get_smoothed_event_signal(
        subject_obj=subject_obj, roi=ACC_ROI, event=EVENT,
        freq_range=FREQ_RANGE, downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS, smooth_method=HFA_SMOOTH_METHOD,
        artifact_abs_thresh=HFA_ARTIFACT_ABS_THRESH,
    )
    hpc_hfa, cue_times_hpc, hpc_labels = _get_smoothed_event_signal(
        subject_obj=subject_obj, roi=HPC_ROI, event=EVENT,
        freq_range=FREQ_RANGE, downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS, smooth_method=HFA_SMOOTH_METHOD,
        artifact_abs_thresh=HFA_ARTIFACT_ABS_THRESH,
    )

    if not np.allclose(np.asarray(cue_times_acc), np.asarray(cue_times_hpc), atol=1e-6, rtol=0):
        raise ValueError('ACC/HPC cue time vectors mismatch')

    n_epochs = int(min(acc_hfa.shape[0], hpc_hfa.shape[0]))
    axis_pack = _build_cue_trial_axis(subject_obj=subject_obj, n_epochs=n_epochs, event=EVENT)
    trial_index = np.asarray(axis_pack['trial_index'], dtype=np.int64)
    trial_number = np.asarray(axis_pack['trial_number'], dtype=np.int64)

    in_range = (trial_index >= 0) & (trial_index < n_epochs)
    trial_index = trial_index[in_range]
    trial_number = trial_number[in_range]
    if trial_index.size == 0:
        raise ValueError('no_valid_trial_index_after_range_filter')

    acc = np.asarray(acc_hfa[trial_index, :, :], dtype=np.float64)
    hpc = np.asarray(hpc_hfa[trial_index, :, :], dtype=np.float64)
    labels_df = _build_behavior_labels_for_subject(subject_obj=subject_obj, n_epochs=n_epochs)

    return {
        'acc': acc,
        'hpc': hpc,
        'cue_times': np.asarray(cue_times_acc, dtype=np.float64),
        'trial_index': trial_index,
        'trial_number': trial_number,
        'acc_labels': np.asarray(acc_labels, dtype=str),
        'hpc_labels': np.asarray(hpc_labels, dtype=str),
        'labels_df': labels_df,
    }


def _run_subject_hfa_frites(subject_id: str) -> Dict[str, Any]:
    try:
        subj = Subject(subject_id, base_dir, load_dir, min_trials=10, verbose=False)
        pack = _prepare_hfa_pair(subj)

        acc = pack['acc']
        hpc = pack['hpc']
        cue_times = pack['cue_times']
        trial_index = pack['trial_index']

        n_trials, n_acc, n_t = acc.shape
        n_trials_h, n_hpc, n_t_h = hpc.shape
        if n_trials_h != n_trials:
            raise ValueError('ACC/HPC trial mismatch after alignment')
        n_t_use = int(min(n_t, n_t_h))

        # Merge hit/miss labels on aligned trial index.
        lab = pack['labels_df'].rename(columns={'epoch_idx': 'trial_index_epoch0'}).copy()
        lab['trial_index_epoch0'] = pd.to_numeric(lab['trial_index_epoch0'], errors='coerce')
        lab = lab.dropna(subset=['trial_index_epoch0'])
        lab['trial_index_epoch0'] = lab['trial_index_epoch0'].astype(int)
        trial_meta = pd.DataFrame({'trial_index_epoch0': trial_index.astype(int)})
        trial_meta = trial_meta.merge(lab[['trial_index_epoch0', 'hit_miss', 'rpe_cat']].drop_duplicates(), on='trial_index_epoch0', how='left')
        hit_miss = trial_meta['hit_miss'].to_numpy(dtype=object)

        est = GCMIEstimator(mi_type='cc', copnorm=True, biascorrect=True)
        rng = np.random.default_rng(SEED + abs(hash(subject_id)) % (2 ** 20))

        rows: List[Dict[str, Any]] = []
        for t in range(n_t_use):
            X = np.asarray(acc[:, :, t], dtype=np.float64)
            Y = np.asarray(hpc[:, :, t], dtype=np.float64)

            mi_all = _safe_mi_cc(est, X, Y)

            hit_idx = np.where(hit_miss == 'hit')[0]
            miss_idx = np.where(hit_miss == 'miss')[0]
            mi_hit = np.nan
            mi_miss = np.nan
            contrast = np.nan
            z_contrast_perm = np.nan
            contrast_null_mean = np.nan
            contrast_null_std = np.nan

            if len(hit_idx) >= MIN_TRIALS_PER_CLASS and len(miss_idx) >= MIN_TRIALS_PER_CLASS:
                mi_hit = _safe_mi_cc(est, X[hit_idx, :], Y[hit_idx, :])
                mi_miss = _safe_mi_cc(est, X[miss_idx, :], Y[miss_idx, :])
                contrast = mi_hit - mi_miss if np.isfinite(mi_hit) and np.isfinite(mi_miss) else np.nan

                labels = np.array(['hit'] * len(hit_idx) + ['miss'] * len(miss_idx), dtype=object)
                XY = np.arange(n_trials)
                use_idx = np.r_[hit_idx, miss_idx]
                null = np.full(int(N_PERM_CONTRAST), np.nan, dtype=float)
                for p in range(int(N_PERM_CONTRAST)):
                    lp = labels[rng.permutation(labels.size)]
                    h = use_idx[lp == 'hit']
                    m = use_idx[lp == 'miss']
                    if len(h) < MIN_TRIALS_PER_CLASS or len(m) < MIN_TRIALS_PER_CLASS:
                        continue
                    mih = _safe_mi_cc(est, X[h, :], Y[h, :])
                    mim = _safe_mi_cc(est, X[m, :], Y[m, :])
                    if np.isfinite(mih) and np.isfinite(mim):
                        null[p] = mih - mim
                null = null[np.isfinite(null)]
                if null.size > 0 and np.isfinite(contrast):
                    contrast_null_mean = float(np.mean(null))
                    contrast_null_std = float(np.std(null))
                    sd = max(contrast_null_std, 1e-12)
                    z_contrast_perm = float((contrast - contrast_null_mean) / sd)

            rows.append({
                'subject': subject_id,
                'time_idx': int(t),
                'time_s': float(cue_times[t]),
                'mi_frites': float(mi_all) if np.isfinite(mi_all) else np.nan,
                'mi_hit_frites': float(mi_hit) if np.isfinite(mi_hit) else np.nan,
                'mi_miss_frites': float(mi_miss) if np.isfinite(mi_miss) else np.nan,
                'hit_minus_miss_mi_frites': float(contrast) if np.isfinite(contrast) else np.nan,
                'contrast_null_mean': float(contrast_null_mean) if np.isfinite(contrast_null_mean) else np.nan,
                'contrast_null_std': float(contrast_null_std) if np.isfinite(contrast_null_std) else np.nan,
                'z_contrast_perm': float(z_contrast_perm) if np.isfinite(z_contrast_perm) else np.nan,
                'n_trials_aligned': int(n_trials),
                'n_hit_trials': int(np.sum(hit_miss == 'hit')),
                'n_miss_trials': int(np.sum(hit_miss == 'miss')),
                'n_acc_electrodes_raw': int(n_acc),
                'n_hpc_electrodes_raw': int(n_hpc),
            })

        return {
            'subject': subject_id,
            'status': 'ok',
            'error': '',
            'traceback': '',
            'rows': rows,
            'labels_df': pack['labels_df'],
            'n_rows': int(len(rows)),
            'n_timepoints': int(n_t_use),
            'n_trials_aligned': int(n_trials),
            'n_acc_electrodes': int(n_acc),
            'n_hpc_electrodes': int(n_hpc),
        }
    except Exception as exc:
        return {
            'subject': subject_id,
            'status': 'error',
            'error': repr(exc),
            'traceback': traceback.format_exc(),
            'rows': [],
            'labels_df': pd.DataFrame(columns=['subject', 'epoch_idx', 'hit_miss', 'rpe_cat']),
            'n_rows': 0,
            'n_timepoints': 0,
            'n_trials_aligned': 0,
            'n_acc_electrodes': 0,
            'n_hpc_electrodes': 0,
        }


In [ ]:

# -----------------------------
# INLINE SMOKE TEST (run first)
# -----------------------------
# Purpose: fail fast on estimator API / return-shape mismatches.
SMOKE_TEST = True

if SMOKE_TEST:
    print('Running inline smoke test...')
    rng = np.random.default_rng(0)

    # Tiny synthetic data with unequal feature counts (ACC vs HPC style).
    n_trials = 24
    n_acc_feat = 5
    n_hpc_feat = 7
    X = rng.normal(size=(n_trials, n_acc_feat))
    Y = rng.normal(size=(n_trials, n_hpc_feat))

    est = GCMIEstimator(mi_type='cc', copnorm=True, biascorrect=True)

    xf = _to_frites_mv(X)
    yf = _to_frites_mv(Y)
    print('Synthetic X/Y:', X.shape, Y.shape)
    print('Frites input xf/yf:', xf.shape, yf.shape)

    out = est.estimate(xf, yf)
    print('Raw est.estimate type:', type(out))
    if isinstance(out, (tuple, list)):
        print('Raw est.estimate length:', len(out))
        for k, v in enumerate(out):
            print(f'  out[{k}] type={type(v)} shape={getattr(v, "shape", None)}')
        out0 = out[0]
    else:
        out0 = out

    mi_scalar = float(np.asarray(out0).squeeze())
    if not np.isfinite(mi_scalar):
        raise RuntimeError('Smoke test failed: non-finite MI scalar from Frites.')
    print('Smoke MI scalar (bits):', mi_scalar)

    # MNE cluster API smoke test (expected 4 returns)
    A = rng.normal(size=(12, 20))
    B = rng.normal(size=(12, 20))
    ret = permutation_cluster_test([A, B], n_permutations=32, tail=1, threshold=None, out_type='indices', seed=0)
    if not isinstance(ret, tuple) or len(ret) != 4:
        raise RuntimeError(f'Smoke test failed: permutation_cluster_test returned {type(ret)} len={getattr(ret, "__len__", lambda: "NA")()} expected tuple len=4')
    T_obs, clusters, cluster_pv, H0 = ret
    print('MNE smoke outputs:', type(T_obs), type(clusters), type(cluster_pv), type(H0))
    print('Inline smoke test passed.')


In [ ]:

# -----------------------------
# Debug: single-subject Frites call inspection (run this first if errors)
# -----------------------------
DEBUG_SUBJECT = 'MS012'  # e.g., 'MS012'
DEBUG_TIME_IDX = 0

if DEBUG_SUBJECT is None:
    print('Set DEBUG_SUBJECT to run single-subject debug.')
else:
    print('Running debug for subject:', DEBUG_SUBJECT)
    subj_dbg = Subject(DEBUG_SUBJECT, base_dir, load_dir, min_trials=10, verbose=False)
    pack_dbg = _prepare_hfa_pair(subj_dbg)
    acc_dbg = pack_dbg['acc']
    hpc_dbg = pack_dbg['hpc']
    print('ACC shape:', acc_dbg.shape, 'HPC shape:', hpc_dbg.shape)

    t = int(DEBUG_TIME_IDX)
    X = np.asarray(acc_dbg[:, :, t], dtype=np.float64)
    Y = np.asarray(hpc_dbg[:, :, t], dtype=np.float64)
    print('Timepoint matrices X/Y:', X.shape, Y.shape)

    est_dbg = GCMIEstimator(mi_type='cc', copnorm=True, biascorrect=True)
    xf = _to_frites_mv(X)
    yf = _to_frites_mv(Y)
    print('Frites input shapes xf/yf:', xf.shape, yf.shape)

    try:
        out = est_dbg.estimate(xf, yf)
        print('Raw est.estimate output type:', type(out))
        if isinstance(out, (tuple, list)):
            print('Tuple/list length:', len(out))
            for k, v in enumerate(out):
                print(f'  out[{k}] type={type(v)} shape={getattr(v, "shape", None)}')
        else:
            print('Output shape:', getattr(out, 'shape', None))
        print('Scalar MI via helper:', _safe_mi_cc(est_dbg, X, Y, debug=True))
    except Exception as exc:
        print('DEBUG ERROR:', repr(exc))
        print(traceback.format_exc())


In [ ]:

# -----------------------------
# Subject list + parallel run (preserved)
# -----------------------------
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

print('Eligible subjects:', len(subjects))
print('First 10 subjects:', subjects[:10])
print('N_JOBS:', N_JOBS, '| backend:', JOBLIB_BACKEND)

results = Parallel(n_jobs=N_JOBS, backend=JOBLIB_BACKEND, verbose=5)(
    delayed(_run_subject_hfa_frites)(sid) for sid in subjects
)

audit_df = pd.DataFrame([{
    'subject': r['subject'],
    'status': r['status'],
    'error': r['error'],
    'traceback': r.get('traceback', ''),
    'n_rows': r['n_rows'],
    'n_timepoints': r['n_timepoints'],
    'n_trials_aligned': r['n_trials_aligned'],
    'n_acc_electrodes': r['n_acc_electrodes'],
    'n_hpc_electrodes': r['n_hpc_electrodes'],
} for r in results])

frames = [pd.DataFrame(r['rows']) for r in results if len(r['rows']) > 0]
hfa_mi_subject_time_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

audit_df.to_csv(output_root / 'subject_audit_frites.csv', index=False)
if len(hfa_mi_subject_time_df):
    hfa_mi_subject_time_df.to_csv(output_root / 'hfa_mi_subject_time_frites.csv', index=False)
    try:
        hfa_mi_subject_time_df.to_parquet(output_root / 'hfa_mi_subject_time_frites.parquet', index=False)
    except Exception:
        pass

print('Requested subjects:', len(subjects))
print('Successful subjects:', int((audit_df['status'] == 'ok').sum()))
print('Failed subjects:', int((audit_df['status'] != 'ok').sum()))
print('Rows:', len(hfa_mi_subject_time_df))
display(audit_df.head(10))
display(hfa_mi_subject_time_df.head(10) if len(hfa_mi_subject_time_df) else hfa_mi_subject_time_df)


In [ ]:

# -----------------------------
# Validation checks
# -----------------------------
if len(hfa_mi_subject_time_df):
    dup = int(hfa_mi_subject_time_df.duplicated(subset=['subject', 'time_idx']).sum())
    print('Duplicate (subject,time_idx):', dup)

    expected = (
        audit_df[audit_df['status'] == 'ok'][['subject', 'n_timepoints']]
        .rename(columns={'n_timepoints': 'expected'})
    )
    observed = hfa_mi_subject_time_df.groupby('subject', as_index=False).size().rename(columns={'size': 'observed'})
    chk = expected.merge(observed, on='subject', how='left').fillna({'observed': 0})
    chk['match'] = chk['expected'].astype(int) == chk['observed'].astype(int)
    display(chk)

    print('Finite MI rows:', int(np.isfinite(hfa_mi_subject_time_df['mi_frites']).sum()), '/', len(hfa_mi_subject_time_df))
    print('Finite z_contrast_perm rows:', int(np.isfinite(hfa_mi_subject_time_df['z_contrast_perm']).sum()), '/', len(hfa_mi_subject_time_df))
else:
    print('No subject-time MI rows available.')


In [ ]:

# -----------------------------
# Hit vs miss cluster permutation test (subject-level time series)
# -----------------------------
if len(hfa_mi_subject_time_df) == 0:
    print('No data for cluster test.')
else:
    ts = (
        hfa_mi_subject_time_df[['subject', 'time_idx', 'time_s', 'hit_minus_miss_mi_frites']]
        .dropna(subset=['hit_minus_miss_mi_frites'])
        .copy()
    )

    if len(ts) == 0:
        print('No finite hit-minus-miss MI rows available for cluster test.')
    else:
        wide = ts.pivot_table(index='subject', columns='time_idx', values='hit_minus_miss_mi_frites', aggfunc='mean').sort_index(axis=1)
        wide = wide.dropna(axis=0, how='any')
        if wide.shape[0] < 2:
            print('Need at least 2 subjects with complete series for cluster test.')
        else:
            tcols = wide.columns.to_numpy(dtype=int)
            tmap = ts[['time_idx', 'time_s']].drop_duplicates().set_index('time_idx')['time_s']
            time_s = np.array([float(tmap.loc[t]) for t in tcols], dtype=float)

            X = wide.to_numpy(dtype=float)
            # one-sample test against 0 with sign-flip permutation via two-sample formulation:
            X0 = np.zeros_like(X)
            T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
                [X, X0],
                n_permutations=int(N_PERM_CLUSTER),
                tail=1,
                threshold=None,
                out_type='indices',
                seed=int(SEED),
            )

            rows=[]
            for cid,(clu,pv) in enumerate(zip(clusters, cluster_pv)):
                tidx = np.asarray(clu[0], dtype=int)
                if tidx.size == 0:
                    continue
                sidx,eidx = int(tidx.min()), int(tidx.max())
                rows.append({
                    'cluster_id': int(cid),
                    'p_value': float(pv),
                    'start_time_s': float(time_s[sidx]),
                    'end_time_s': float(time_s[eidx]),
                    'n_timepoints': int(tidx.size),
                    'cluster_stat': float(np.sum(T_obs[tidx])),
                    'sign': 'hit>miss',
                    'is_sig': bool(float(pv) < float(CLUSTER_ALPHA)),
                })
            cluster_df = pd.DataFrame(rows)
            if len(cluster_df) == 0:
                cluster_df = pd.DataFrame([{
                    'cluster_id': -1, 'p_value': np.nan, 'start_time_s': np.nan,
                    'end_time_s': np.nan, 'n_timepoints': 0, 'cluster_stat': np.nan,
                    'sign': 'hit>miss', 'is_sig': False,
                }])

            cluster_df.to_csv(output_root / 'hit_vs_miss_cluster_test_frites.csv', index=False)
            np.savez(output_root / 'hit_vs_miss_cluster_test_frites_raw.npz',
                     T_obs=np.asarray(T_obs, dtype=float),
                     cluster_pv=np.asarray(cluster_pv, dtype=float),
                     H0=np.asarray(H0, dtype=float),
                     time_idx=np.asarray(tcols, dtype=int),
                     time_s=np.asarray(time_s, dtype=float),
                     mean_diff=np.nanmean(X, axis=0))

            fig,ax = plt.subplots(1,1,figsize=(10,4))
            mean_diff = np.nanmean(X, axis=0)
            ax.plot(time_s, mean_diff, lw=2, color='tab:blue', label='mean (hit-miss) MI')
            ax.axhline(0, color='k', ls='--', lw=1)
            sig_count=0
            for clu,pv in zip(clusters, cluster_pv):
                tidx=np.asarray(clu[0],dtype=int)
                if tidx.size and float(pv) < float(CLUSTER_ALPHA):
                    sig_count += 1
                    ax.axvspan(time_s[int(tidx.min())], time_s[int(tidx.max())], color='tomato', alpha=.2)
            ax.set_title(f'Subject-level cluster test on hit-miss MI (Frites, Nperm={int(N_PERM_CLUSTER)})')
            ax.set_xlabel('Cue time (s)')
            ax.set_ylabel('Hit - Miss MI (bits)')
            ax.legend(frameon=False)
            fig.tight_layout()
            fig.savefig(fig_root / 'hit_vs_miss_cluster_test_frites.png', dpi=200, bbox_inches='tight')
            plt.show()

            print('Cluster rows:', len(cluster_df), '| significant:', int(np.sum(cluster_df['is_sig'].astype(bool))))
            display(cluster_df)


In [ ]:

# -----------------------------
# Export per-subject effect in significant cluster-window union
# -----------------------------
cluster_csv = output_root / 'hit_vs_miss_cluster_test_frites.csv'
export_dir = Path('/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss')
export_csv = export_dir / 'ACC_HPC_memory_outcome_hit_miss_hfa_subject_mi_cluster_effect.csv'
export_meta_csv = export_dir / 'ACC_HPC_memory_outcome_hit_miss_hfa_subject_mi_cluster_effect_metadata.csv'

if len(hfa_mi_subject_time_df) == 0 or (not cluster_csv.exists()):
    print('Missing required inputs for export.')
else:
    cdf = pd.read_csv(cluster_csv)
    if 'is_sig' not in cdf.columns:
        raise ValueError('cluster csv missing is_sig')

    def _to_bool(x):
        if pd.isna(x):
            return False
        return str(x).strip().lower() in {'true', '1', 'yes'}

    sig = cdf[cdf['is_sig'].map(_to_bool)].copy()
    subj_all = sorted(hfa_mi_subject_time_df['subject'].dropna().astype(str).unique().tolist())

    if len(sig) == 0:
        out = pd.DataFrame({'Subject': subj_all, 'MeanAccuracy': np.nan, 'ReinstateEffect': np.nan})
    else:
        use = hfa_mi_subject_time_df.copy()
        in_union = np.zeros(len(use), dtype=bool)
        sigw = sig[['start_time_s', 'end_time_s']].dropna().copy()
        for _,r in sigw.iterrows():
            lo,hi = sorted([float(r['start_time_s']), float(r['end_time_s'])])
            in_union |= (use['time_s'].to_numpy(dtype=float) >= lo) & (use['time_s'].to_numpy(dtype=float) <= hi)
        use = use.loc[in_union].copy()

        hit = use.groupby('subject', as_index=False)['mi_hit_frites'].mean().rename(columns={'mi_hit_frites': 'MeanAccuracy'})
        miss = use.groupby('subject', as_index=False)['mi_miss_frites'].mean().rename(columns={'mi_miss_frites': 'MissMean'})
        out = pd.DataFrame({'Subject': subj_all})
        out = out.merge(hit.rename(columns={'subject': 'Subject'}), on='Subject', how='left')
        out = out.merge(miss.rename(columns={'subject': 'Subject'}), on='Subject', how='left')
        out['ReinstateEffect'] = out['MeanAccuracy'] - out['MissMean']
        out = out.drop(columns=['MissMean'])

    elec = hfa_mi_subject_time_df.groupby('subject', as_index=False).agg(n_acc=('n_acc_electrodes_raw','max'), n_hpc=('n_hpc_electrodes_raw','max'))
    elec['NumElectrodes'] = elec['n_acc'].fillna(0) + elec['n_hpc'].fillna(0)
    out = out.merge(elec[['subject','NumElectrodes']].rename(columns={'subject':'Subject'}), on='Subject', how='left')

    export_dir.mkdir(parents=True, exist_ok=True)
    out[['Subject','MeanAccuracy','ReinstateEffect','NumElectrodes']].to_csv(export_csv, index=False)

    meta = sig.copy()
    meta['cluster_alpha'] = float(CLUSTER_ALPHA)
    meta['n_perm_cluster'] = int(N_PERM_CLUSTER)
    meta['seed'] = int(SEED)
    meta.to_csv(export_meta_csv, index=False)

    print('Saved:', export_csv)
    print('Saved:', export_meta_csv)
    display(out.head(15))



## Notes
- MI is computed with Frites Gaussian-copula MI (`GCMIEstimator`, `mi_type='cc'`) in **bits**.
- Primary table is subject-time: `hfa_mi_subject_time_df`.
- Cluster test and export are based on subject-level hit-minus-miss MI trajectories.
